**String Composition Problem: Generate the k-mer composition of a string.**

In [ ]:
def string_composition(Text, k):
    kmers = []
    for i in range(len(Text) - k + 1):
        kmers.append(Text[i:i+k])
        
    return kmers # lexicographical order - sorted(kmers)

**Reconstruct a string from its genome path.**

In [ ]:
def PathtoGenome(path):
    kmers = path.split()
    genome = kmers[0]
    for kmer in kmers[1:]:
        genome += kmer[-1]
    return genome

**Overlap Graph Problem**

In [ ]:
def OverlapGraph(nodes):
    kmers = nodes.split()
    directed_graph = {}

    for i in kmers:
        suffix = []
        for j in kmers:
            if i != j:
                if i[1:] == j[:-1]:
                    suffix.append(j)
        if suffix:
            directed_graph[i] = suffix

    return directed_graph

**De Bruijn Graph from a String Problem**

In [ ]:
# defaultdict automatically assign a default value to a key that does not exist, 
# thereby eliminating KeyError exceptions

# 1. Sort the dictionary items ---- sorted
# 2. Store them while preserving that order ---- OrderedDict

from collections import defaultdict, OrderedDict

def DeBruijnGraph_from_string(Text, k):
  de_bruijn_graph = defaultdict(list)
  for i in range(len(Text) - k + 1):
    kmer = Text[i:i+k]
    prefix = kmer[:k-1]
    suffix = kmer[1:]
    de_bruijn_graph[prefix].append(suffix)
    
  return OrderedDict(sorted(de_bruijn_graph.items()))

**DeBruijn Graph from k-mers Problem**

In [ ]:
from collections import defaultdict, OrderedDict

def DeBruijnGraph_from_kmers(kmers):
  de_bruijn_graph = defaultdict(list)
  for i in kmers.split():
    prefix = i[:len(i)-1]
    suffix = i[1:]
    de_bruijn_graph[prefix].append(suffix)
    
  return OrderedDict(sorted(de_bruijn_graph.items()))

**Eulerian Cycle Problem**

In [ ]:
import random

def read_adjacency_list(filename):
    graph = {}
    with open(filename, 'r') as file:
        for line in file:
            parts = line.strip().split(':')
            node = int(parts[0])
            neighbors = list(map(int, parts[1].split()))
            graph[node] = neighbors
    return graph

def eulerian_cycle(graph):
    if not graph:
        raise ValueError("The graph is empty.")

    current_node = random.choice(list(graph.keys()))
    path = [current_node]
    
    while True:
        if current_node not in graph or not graph[current_node]:
            break
        next_node = graph[current_node][0]
        path.append(next_node)
        
        if len(graph[current_node]) == 1:
            del graph[current_node] # so that every edge is traversed once
        else:
            graph[current_node] = graph[current_node][1:]
        
        current_node = next_node

    while len(graph) > 0:
        for i in range(len(path)):
            if path[i] in graph:
                current_node = path[i]
                cycle = [current_node]
                while True:
                    if current_node not in graph or not graph[current_node]:
                        break
                    next_node = graph[current_node][0]
                    cycle.append(next_node)

                    if len(graph[current_node]) == 1:
                        del graph[current_node]
                    else:
                        graph[current_node] = graph[current_node][1:]
                    
                    current_node = next_node

                path = path[:i] + cycle + path[i+1:]
                break

    return path

**Eulerian Path Problem**

In [ ]:
import random

def check_balance(graph):
    incoming_edges = 0
    outgoing_edges = 0
    start_nodes = []
    for i, j in graph.items():
        outgoing_edges = len(j)
        for k in graph.values():
            if i in k:
                incoming_edges += 1
        if outgoing_edges > incoming_edges:
            start_nodes.append(i)
        incoming_edges = 0

    return start_nodes

def eulerian_path(graph):
    if not graph:
        raise ValueError("The graph is empty.")

    start_nodes = check_balance(graph)
    if len(start_nodes) > 0:
        print("The graph is unbalanced and will form an Eulerian Path, not an Eulerian Cycle")
        current_node = random.choice(start_nodes)
        path = [current_node]
        
        while True:
            if current_node not in graph or not graph[current_node]:
                break
            next_node = graph[current_node][0]
            path.append(next_node)
            
            if len(graph[current_node]) == 1:
                del graph[current_node]
            else:
                graph[current_node] = graph[current_node][1:]
            
            current_node = next_node

        while len(graph) > 0:
            for i in range(len(path)):
                if path[i] in graph:
                    current_node = path[i]
                    cycle = [current_node]
                    while True:
                        if current_node not in graph or not graph[current_node]:
                            break
                        next_node = graph[current_node][0]
                        cycle.append(next_node)

                        if len(graph[current_node]) == 1:
                            del graph[current_node]
                        else:
                            graph[current_node] = graph[current_node][1:]
                        
                        current_node = next_node

                    path = path[:i] + cycle + path[i+1:]
                    break

        return path
    else:
        print("The graph is balanced and will form an Eulerian Cycle, not an Eulerian Path")
        return eulerian_cycle(graph)

**Generating a k-universal binary string: linear or circular**

In [ ]:
from itertools import product


def debruijn(k):
    # Generate all binary (k-1)-mers
    nodes = [''.join(p) for p in product('01', repeat=k-1)]
    graph = {}
    for node in nodes:
        graph[node] = [
            node[1:] + '0',
            node[1:] + '1'
        ]

    # Eulerian cycle (Hierholzer's algorithm)
    stack = [nodes[0]]
    path = []

    while stack:
        node = stack[-1]
        if graph[node]:
            stack.append(graph[node].pop()) # every edge is traversed once
        else:
            path.append(stack.pop()) # every node with no outgoing edges are appended

    path.reverse() # backtracking
    return path


def k_universal(k, circular=False):
    cycle = debruijn(k)
    if circular:
        seq = cycle[0]
        for node in cycle[1:-k+1]: 
        # Removing the last two nodes so that the ending does not overlap the beginning
            seq += node[-1]
        return seq
    else:
        seq = cycle[0]
        for node in cycle[1:]:
            seq += node[-1]
        return seq[:2**k + k - 1]

def count_k_universal(k):
    circular = 2 ** (2**(k-1) - k) # BEST Theorem
    linear = 2 ** (2**(k-1))

    return {
        "circular": circular,
        "linear": linear
    }

# A circular sequence has 2**k different positions where it can be cut.
# Because there are exactly 2**k distinct k-mers around the cycle,
# N(linear) = 2**k * N(circular)